In [1]:
import pandas as pd
import time
from geopy.geocoders import Nominatim

# Load CSV files
df_konvensional = pd.read_csv('/Users/mraffyzeidan/Learning/xcap/Client-BPRSK/ojk_cfs_bpr_konvensional.csv')
df_syariah = pd.read_csv('/Users/mraffyzeidan/Learning/xcap/Client-BPRSK/ojk_cfs_bpr_syariah.csv')

print("Data Konvensional shape:", df_konvensional.shape)
print("Data Syariah shape:", df_syariah.shape)
print("\nKolom:", df_konvensional.columns.tolist())

Data Konvensional shape: (1863, 4)
Data Syariah shape: (194, 4)

Kolom: ['Provinsi', 'Kabupaten/Kota', 'Nama Bank', 'Kode Bank']


In [2]:
import pandas as pd
from geopy.geocoders import Nominatim
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

# Load CSV files
print('Loading CSV files...')
df_konvensional = pd.read_csv('/Users/mraffyzeidan/Learning/xcap/Client-BPRSK/ojk_cfs_bpr_konvensional.csv')
df_syariah = pd.read_csv('/Users/mraffyzeidan/Learning/xcap/Client-BPRSK/ojk_cfs_bpr_syariah.csv')

print(f"Data Konvensional: {df_konvensional.shape}")
print(f"Data Syariah: {df_syariah.shape}")

# Extract unique banks
banks_konvensional = df_konvensional[['Nama Bank', 'Provinsi', 'Kabupaten/Kota']].drop_duplicates()
banks_syariah = df_syariah[['Nama Bank', 'Provinsi', 'Kabupaten/Kota']].drop_duplicates()

all_banks = pd.concat([
    banks_konvensional.assign(Jenis='Konvensional'),
    banks_syariah.assign(Jenis='Syariah')
], ignore_index=True)

print(f"Total bank konvensional unik: {len(banks_konvensional)}")
print(f"Total bank syariah unik: {len(banks_syariah)}")
print(f"Total bank unik: {len(all_banks)}")

Loading CSV files...
Data Konvensional: (1863, 4)
Data Syariah: (194, 4)
Total bank konvensional unik: 1861
Total bank syariah unik: 194
Total bank unik: 2055


In [3]:
# Geocode nama bank spesifik dengan THREAD POOL (concurrent)

def get_bank_coordinates(bank_name, city_name, province_name, timeout=10, max_retries=3):
    from geopy.geocoders import Nominatim
    import time
    
    # Try Nominatim first (free)
    for attempt in range(max_retries):
        try:
            geolocator = Nominatim(user_agent="bank_locator", timeout=timeout)
            location_str = f"{bank_name}, {city_name}, {province_name}, Indonesia"
            location = geolocator.geocode(location_str)
            if location:
                return location.latitude, location.longitude, "SUCCESS_BANK"
            break
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
    
    # Fallback ke city center kalau bank name gabisa di-geocode
    try:
        geolocator = Nominatim(user_agent="bank_locator", timeout=timeout)
        location = geolocator.geocode(f"{city_name}, Indonesia")
        if location:
            import random
            lat = location.latitude + random.uniform(-0.01, 0.01)
            lon = location.longitude + random.uniform(-0.01, 0.01)
            return lat, lon, "SUCCESS_CITY"
    except:
        pass
    
    return None, None, "FAILED"

from concurrent.futures import ThreadPoolExecutor, as_completed

print(f"\nGeocoding {len(all_banks)} bank dengan THREAD POOL (max_workers=5)...\n")

all_banks['Latitude'] = None
all_banks['Longitude'] = None

berhasil_count = 0
gagal_count = 0
max_workers = 4  # Adjust ini kalau mau lebih cepat/lambat

results = {}

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    # Submit all tasks
    futures = {}
    for idx, row in all_banks.iterrows():
        future = executor.submit(
            get_bank_coordinates,
            row['Nama Bank'],
            row['Kabupaten/Kota'],
            row['Provinsi']
        )
        futures[future] = idx
    
    # Process hasil sebagai selesai
    for future in tqdm(as_completed(futures), total=len(futures)):
        idx = futures[future]
        lat, lon, status = future.result()
        results[idx] = (lat, lon, status)
        
        row = all_banks.iloc[idx]
        
        # DEBUG: Print tiap proses
        if status == "SUCCESS_BANK":
            berhasil_count += 1
            print(f"✓ [{berhasil_count}] BANK: {row['Nama Bank'][:40]} | {row['Kabupaten/Kota']} | ({lat:.4f}, {lon:.4f})")
        elif status == "SUCCESS_CITY":
            berhasil_count += 1
            print(f"✓ [{berhasil_count}] CITY: {row['Nama Bank'][:40]} | {row['Kabupaten/Kota']} | ({lat:.4f}, {lon:.4f})")
        else:
            gagal_count += 1
            if gagal_count <= 20:  # Print hanya 20 yang gagal pertama
                print(f"✗ [{gagal_count}] FAIL: {row['Nama Bank'][:40]} | {row['Kabupaten/Kota']}")

# Assign hasil ke dataframe (sorted by index)
for idx, (lat, lon, status) in sorted(results.items()):
    all_banks.at[idx, 'Latitude'] = lat
    all_banks.at[idx, 'Longitude'] = lon

print(f"\n{'='*70}")
print(f"FINAL RESULT (dengan {max_workers} threads):")
print(f"{'='*70}")
print(f"Total: {len(all_banks)}")
print(f"Berhasil: {berhasil_count}")
print(f"Gagal: {gagal_count}")
print(f"Success rate: {berhasil_count/len(all_banks)*100:.1f}%")
print(f"{'='*70}\n")


Geocoding 2055 bank dengan THREAD POOL (max_workers=5)...



  0%|          | 1/2055 [00:05<2:51:50,  5.02s/it]

✓ [1] CITY: PT. BPR Siwa Raharja Utama | Kab. Bekasi | (-6.1940, 107.1594)


  0%|          | 2/2055 [00:05<1:29:19,  2.61s/it]

✓ [2] CITY: PT Bank Perekonomian Rakyat Dana Multi G | Kab. Bekasi | (-6.1978, 107.1643)


...

✓ [1605] CITY: PT. BPRS Muamalat Yotefa | Kab. Jayapura | (-2.5508, 140.4884)


100%|█████████▉| 2054/2055 [1:08:04<00:01,  1.80s/it]

✓ [1606] CITY: PT Bank Perekonomian Rakyat Syariah Baha | Kota Ternate | (0.7774, 127.3862)


100%|██████████| 2055/2055 [1:08:05<00:00,  1.99s/it]

✓ [1607] CITY: PT Bank Perekonomian Rakyat Syariah Boba | Kota Tidore Kepulauan | (0.6690, 127.4440)

FINAL RESULT (dengan 4 threads):
Total: 2055
Berhasil: 1607
Gagal: 448
Success rate: 78.2%



In [6]:
all_banks.to_csv('BankLongLat.csv', index=False)